In [2]:
import torch 
import torch.nn as nn

In [3]:
class PatchEmbedding(nn.Module):
  def __init__(self,in_channels,patch_size,embed_dim):
    super().__init__()
    self.patch_emb=nn.Conv2d(in_channels,embed_dim,kernel_size=patch_size,stride=patch_size)

  def forward(self,x):
    b,c,h,w=x.shape
    x=self.patch_emb(x) # 32,768,14, 14

    x=x.flatten(2)   #32,768, 196
    x=x.transpose(1,2) # 32,196,768
    return x

In [4]:
class PositionalEncoding(nn.Module):
    def __init__(self, embed_dim, seq_len):
        super().__init__()
        self.pos_embed = nn.Parameter(torch.randn(1, seq_len, embed_dim))  #1,197,768

    def forward(self, x):
        return x + self.pos_embed  #32 , 197,768

In [5]:
class TransformerEncoderBlock(nn.Module):
    def __init__(self, embed_dim, num_heads, mlp_dim, dropout):
        super().__init__()
        self.block = nn.TransformerEncoderLayer(
            d_model=embed_dim,    # 768
            nhead=num_heads,      # 8
            dim_feedforward=mlp_dim,
            dropout=dropout,
            batch_first=True,  # ← IMPORTANT!
            activation='gelu',
            norm_first=True
        )

    def forward(self, x):
        return self.block(x)   # 32,197,768

In [6]:
class C3TR(nn.Module):
    def __init__(self, in_channels, patch_size, embed_dim, patches,
                 mlp_dim, num_heads, dropout, depth):
        super().__init__()

        # patch embedding
        self.patch_embed = PatchEmbedding(in_channels, patch_size, embed_dim)

        # positional encoding (no CLS now)
        self.pos_embed = PositionalEncoding(embed_dim, patches)

        # transformer blocks
        self.transformer_blocks = nn.ModuleList([
            TransformerEncoderBlock(embed_dim, num_heads, mlp_dim, dropout)
            for _ in range(depth)
        ])

        # project back to original channels if embed_dim != in_channels
        self.proj_back = nn.Linear(embed_dim, in_channels)

    def forward(self, x):
        B, C, H, W = x.shape
        residual = x                               # (B, C, H, W)

        # Patch embedding
        x = self.patch_embed(x)                    # (B, N, embed_dim)

        # Positional encoding
        x = self.pos_embed(x)                      # (B, N, embed_dim)

        # Transformer
        for block in self.transformer_blocks:
            x = block(x)                           # (B, N, embed_dim)

        # Project back to original channel size
        x = self.proj_back(x)                      # (B, N, C)

        # Reshape back to feature map
        x = x.transpose(1, 2)                      # (B, C, N)
        x = x.reshape(B, C, H, W)                  # (B, C, H, W)

        # Residual connection
        x = x + residual                           # (B, C, H, W)

        return x

In [7]:
c3tr = C3TR(
    in_channels=960,
    patch_size=1,
    embed_dim=960,
    patches=20*20,
    mlp_dim=960*4,
    num_heads=8,
    dropout=0.1,
    depth=3
)

In [8]:
# dummy
import torch

B = 2
dummy = torch.randn(B, 960, 20, 20)

out = c3tr(dummy)

print("Input:", dummy.shape)
print("Output:", out.shape)

Input: torch.Size([2, 960, 20, 20])
Output: torch.Size([2, 960, 20, 20])
